# 02 · crop 생성

샘플 단위는 crop 이다 (D-01). geometry 는 `configs/crop.yaml` 의 `crop-context-2.0` 이며
기존 저장소에서 QC 100장 중 8건 실패로 통과한 조합이다 (D-15).

## 대상

```
selector = head_a6_eligible | head_a6_aux_pool | head_a7_aux_train
         + legacy_annotation_error (review-only)
         - sealed_future_eval        (이미지 I/O 자체 금지)
```

| lane | 데이터셋 | 처리 |
| --- | --- | --- |
| baseline | AIHub 1,546 | polygon → crop |
| baseline | DACON 3,448 | identity (평가 후보 3,352 + train-only 96) |
| baseline | Kaggle 1,500 | identity |
| ablation | RHD 2,217 · RWKR 741 · RWD 211 | bbox → crop |
| lineage_a7 | RWD moisture 103 | bbox → crop |
| review_only | AIHub 15 | crop 만 생성, 학습·평가 금지 |
| (제외) | sealed 45 | 이미지를 열지 않는다 |

**Roboflow 는 baseline 에 안 들어가지만 지금 만든다.** A9 arm 과 head_a7 계보가
이 데이터에 의존하므로, 미루면 나중에 crop 파이프라인을 다시 열어야 한다.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd

pd.set_option("display.width", 170)
from banggoot import crops, metadata

cfg = crops.load_crop_config()
cfg["geometry"]

{'context_ratio': 0.35,
 'min_crop_px': 128,
 'max_crop_fraction': 0.55,
 'min_retained_area': 0.5,
 'dedupe_iou': 0.7,
 'square_target': True,
 'legacy_rounding_tolerance_px': 1}

## 1. geometry — cap 과 floor는 짝이다

```python
side = max(bw, bh) * (1 + 2*context_ratio)
side = max(side, min_crop_px)
side = min(side, short_edge * max_crop_fraction)   # 0.55 cap
side = max(side, min(max(bw, bh), short_edge))     # floor — seed box 아래로 못 내려감
side = min(side, short_edge)
```

**floor 없이 cap만 쓰면 legacy 산출물이 재현되지 않는다.**
`0.55 × 1080 = 594` 를 넘는 crop 이 411건 있는데, cap 미적용이 아니라 seed box 장변이
cap 보다 커서 floor 가 다시 올린 결과다 (D-15).

| `max_crop_fraction` | legacy 2,783건 재현율 (±1px) |
| ---: | ---: |
| **0.55** | **100.00%** |
| 1.00 | 75.35% |

In [2]:
df = metadata.build()
parents = crops.select_parents(df)
parents["lane"] = [crops.lane_of(r) for r in parents.itertuples()]

print(f"crop 대상 parent : {len(parents):,}")
print(f"sealed 포함      : {int(parents.sealed_future_eval.sum())}  (0 이어야 함)")
display(pd.crosstab(parents.lane, parents.dataset, margins=True))

crop 대상 parent : 9,781
sealed 포함      : 0  (0 이어야 함)


dataset,aihub_567,dacon_wallpaper,kaggle_cracks,roboflow_house_defect,roboflow_wall_defects,roboflow_wallpaper_kr,All
lane,,,,,,,
ablation,0,0,0,2217,211,741,3169
baseline,1546,3448,1500,0,0,0,6494
lineage_a7,0,0,0,0,103,0,103
review_only,15,0,0,0,0,0,15
All,1561,3448,1500,2217,314,741,9781


## 2. 생성

AIHub polygon 은 승계한 `annotations.csv`(3,724 box)를 쓴다. Roboflow 는 YOLO txt 를
파싱하고 `data.yaml` 의 `names` 인덱스로 L1 을 매핑한다.

각 crop 에 **seed bbox 좌표와 `source_annotation_index`** 를 기록한다.
`sample_id` 의 `#i` 는 dedupe **이후** 순번이라 원본 annotation 순번과 다르다 —
이 둘을 혼동하면 회귀검증이 잘못된 쌍을 비교한다.

In [3]:
c, stats = crops.build(df)

print(f"parents seen      : {stats.parents_seen:,}")
print(f"crops written     : {stats.crops_written:,}")
print(f"identity samples  : {stats.identity:,}")
print(f"manifest rows     : {len(c):,}")
print(f"skipped no_box    : {stats.skipped_no_box}")
print(f"skipped unreadable: {stats.skipped_unreadable}")
print(f"mixed L1 in crop  : {stats.mixed_l1}")

crops:   0%|          | 0/9781 [00:00<?, ?it/s]

parents seen      : 9,781
crops written     : 8,316
identity samples  : 4,948
manifest rows     : 13,264
skipped no_box    : 0
skipped unreadable: 0
mixed L1 in crop  : 0


In [4]:
display(pd.crosstab(c.lane, c.dataset, margins=True))
display(pd.crosstab(c[c.lane.eq("baseline")].unified_label,
                    c[c.lane.eq("baseline")].dataset, margins=True))

dataset,aihub_567,dacon_wallpaper,kaggle_cracks,roboflow_house_defect,roboflow_wall_defects,roboflow_wallpaper_kr,All
lane,,,,,,,
ablation,0,0,0,3159,276,970,4405
baseline,3691,3448,1500,0,0,0,8639
lineage_a7,0,0,0,0,205,0,205
review_only,15,0,0,0,0,0,15
All,3706,3448,1500,3159,481,970,13264


dataset,aihub_567,dacon_wallpaper,kaggle_cracks,All
unified_label,,,,
breakage,1848,1817,0,3665
crack,1843,0,1500,3343
finish_damage,0,799,0,799
lifting,0,76,0,76
mold,0,144,0,144
stain_corrosion,0,612,0,612
All,3691,3448,1500,8639


### `mixed_l1_in_crop = 0` 은 버그가 아니다

crop 대상 parent 중 실제로 여러 L1 을 가진 이미지가 없다.
AIHub 0/1,561 · RHD 표본 400/400 이 단일 L1 이고, RWD 의 다중 라벨 11건은
`is_multilabel` 로 metadata 단계에서 이미 제외됐다 (D-09).

In [5]:
b = crops.load_aihub_boxes()
multi = sum(1 for v in b.values() if len({x.label for x in v}) > 1)
print(f"AIHub parent 중 다중 L1 보유: {multi} / {len(b)}")
print(f"metadata 에서 is_multilabel 로 제외: {int(df.is_multilabel.sum())}")

AIHub parent 중 다중 L1 보유: 0 / 1561
metadata 에서 is_multilabel 로 제외: 11


## 3. 회귀검증 — 기존 2,783 crop

기존 산출물은 **기준으로만** 쓰고 승계하지 않는다 (D-15 규칙 7).
매칭 키는 **seed bbox 좌표 전체**(cx, cy, w, h)이며 순번을 추론하지 않는다.

legacy 는 `legacy_split=train` + `severity in (normal, poor)` 만 대상이었으므로
신규가 더 많은 것이 정상이다.

In [6]:
import subprocess

r = subprocess.run([sys.executable, str(REPO_ROOT / "scripts" / "regress_crops.py")],
                   capture_output=True, text=True, encoding="utf-8", cwd=REPO_ROOT)
print(r.stdout)
assert r.returncode == 0, r.stderr

Exception in thread Thread-3 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "C:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xbd in position 35: invalid start byte


None


## 4. 최종 gate

- 계보가 겹치지 않는다 (`aux_pool ∩ a7_aux = ∅`, `baseline ∩ aux_pool = ∅`)
- `sealed_future_eval` 행의 crop 이 존재하지 않는다
- DACON identity = **3,448** (평가 후보 3,352 아님)
- baseline 이 아닌 lane 은 `baseline_eligible=False`
- 정사각 100% · `seed_box_id` / `sample_id` 유일
- 선택된 parent 전부가 산출물에 존재
- `skipped_no_box = 0` · `skipped_unreadable = 0`
- **manifest 참조 파일 집합 == 실제 디스크 파일 집합**

In [7]:
crops.verify(c, parents, df)
print("manifest gate 통과")

crops.verify_files(c, stats)
print("파일 gate 통과")

manifest gate 통과


파일 gate 통과


In [8]:
out = crops.save(c)
print("saved:", out)
print(f"  {len(c):,} rows x {len(c.columns)} cols")
print("\n다음: 03_eda_quality")

saved: C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\banggoot_model\artifacts\metadata\crops.csv
  13,264 rows x 32 cols

다음: 03_eda_quality
